# Habitat suitability under climate change

Our changing climate is changing where plant species can live,
and conservation and restoration practices will need to take
this into
account.

In this coding challenge, you will create a habitat suitability model
for a terrestrial plant species of your choice that lives in the contiguous United States
(CONUS). We have this limitation because the downscaled climate data we
suggest, the [MACAv2 dataset](https://www.climatologylab.org/maca.html),
is only available in the CONUS – if you find other downscaled climate
data at an appropriate resolution, you are welcome to choose a different
study area. If you don’t have anything in mind, you can take a look at
[*Sorghastrum nutans*](https://www.gbif.org/species/2704414), a grass native to North America. In the past 50
years, its range has moved
northward.

Your suitability assessment will be based on combining multiple data
layers related to soil, topography, and climate, then applying a fuzzy logic model across the different data layers to generate habitat suitability maps. 

You will need to create a **modular, reproducible, workflow** using functions and loops.
To do this effectively, we recommend planning your code out in advance
using a technique such as a pseudocode outline or a flow diagram. We
recommend breaking each of the blocks below out into multiple steps. It
is unnecessary to write a step for every line of code unless you find
that useful. As a rule of thumb, aim for steps that cover the major
structures of your code in 2-5 line chunks.

In [1]:
# To create reproducible file paths
import os
from pathlib import Path
import pathlib

In [2]:
# To use APIs
import requests

# To download the GBIF data
from getpass import getpass
import pygbif.occurrences as occ
import pygbif.species as species

In [3]:
# For unzipping folders
import time
import zipfile

In [4]:
# To work with different types of data
import geopandas as gpd # To make GeoDataFrames/work with vector data
from glob import glob # To combine data arrays
import numpy as np # To work with arrays
import pandas as pd # To work with dataframes
import rioxarray as rxr # To work with raster data

In [5]:
import rioxarray.merge as rxrm # To merge raster data
from shapely.geometry import MultiPolygon, Polygon # To handle invalid geometries
import xrspatial # To handle spatial data

In [6]:
# For visualization and interactive plotting
import geoviews as gv
import holoviews as hv
import hvplot.pandas
import hvplot.xarray

In [ ]:
# Import the below libraries:
# To create reproducible file paths
import os
from pathlib import Path
import pathlib

# To use APIs
import requests

# To download the GBIF data
from getpass import getpass
import pygbif.occurrences as occ
import pygbif.species as species

# For unzipping folders
import time
import zipfile

# To work with different types of data
import fiona # To view different layers in a shapefile
import geopandas as gpd # To make GeoDataFrames/work with vector data
from glob import glob # To combine data arrays
import numpy as np # To work with arrays
import pandas as pd # To work with dataframes
import rioxarray as rxr # To work with raster data
import rioxarray.merge as rxrm # To merge raster data
from shapely.geometry import MultiPolygon, Polygon # To handle invalid geometries
import xrspatial # To handle spatial data

# For visualization and interactive plotting
import geoviews as gv
import holoviews as hv
import hvplot.pandas
import hvplot.xarray

In [8]:
# Create a designated folder for the repository data
data_dir = os.path.join(
    pathlib.Path.home(),
    # In the earth-analytics data folder
    'earth-analytics',
    'data',
    # Specify the destination as inside the "spring-03-habitat-suitability-climate-change-_____" repository
    'spring-03-habitat-suitability-climate-change-Livian-Von-Dran',
    'redwood_habitat_suitability'
)

# Create the directory
os.makedirs(data_dir, exist_ok=True)

## STEP 1: Study overview

Before you begin coding, you will need to design your study.

### Step 1a: Select a species
Select the terrestrial plant species you want to study, and research its habitat parameters in scientific studies or other reliable sources. Individual studies may not have the breadth needed for this purpose, so take a look at reviews or overviews of the data. Do **not** just look at an AI-generated summary! In the US, the National Resource Conservation Service can have helpful fact sheets about different species. University Extension programs are also good resources for summaries.</p>
<p>Based on your research, select soil, topographic, and climate variables that you can use to determine if a particular location and time period is a suitable habitat for your species.</p></div></div>

**Reflect and respond**: 
Write a description of your species. What habitat is it found in? What is its geographic range? What, if any, are conservation threats to the species? What data will shed the most light on habitat suitability for this species? 

What core scientific question do you hope to answer about potential future changes in habitat suitability? Don't forget to cite your sources!

Your response here:

The species I have chosen is known by several names, chief among them being the California redwood or coast redwood (*Sequoia sempervirens*). Belonging to the Sequoioideae, this species is among the largest tree species ever documented, making it a desirable target for logging operations. Historically, redwood forests were restricted to a 900 kilometer belt along the Coast Range that spanned between central California and southern Oregon (Lorimer et al., 2009), but logging has removed 95% of the existing old-growth forests (Save the Redwoods League, 2018). Remaining old-growth redwood forests continue to be threatened by deforestation, invasive species, and increasingly destructive wildfires (Lorimer et al., 2009). Given that a previous project measured fog occurrence to support redwood habitat assessments, such data may be useful to determine habitat suitability for the California redwood (Wernet et al., 2020). 

References

Burns, E. E., Campbell, R., & Cowan, P. D. (2018). *State of Redwoods Conservation Report*. https://www.savetheredwoods.org/wp-content/uploads/State-of-Redwoods-Conservation-Report-Final-web.pdf

Lorimer, C. G., Porter, D. J., Madej, M. A., Stuart, J. D., Veirs, S. D., Norman, S. P., O’Hara, K. L., & Libby, W. J. (2009). Presettlement and modern disturbance regimes in coast redwood forests: Implications for the conservation of old-growth stands. *Forest Ecology and Management*, *258*(7), 1038–1054. https://doi.org/10.1016/j.foreco.2009.07.008

Werner, Z., Berger, A., Winter, A., Choi, C. T. H., Evangelista, P., Jarnevich, C., Vorster, T., Woodward, B., & Young N. (2020). *California & Oregon Ecological Forecasting: Detecting and Forecasting Fog Occurrence, Frequency, and Change to Support Coast Redwood (Sequoia sempervirens) Habitat Assessments*. https://ntrs.nasa.gov/api/citations/20205011382/downloads/2020Fall_CO_California%26OregonEco_ProjectSummary_FD-final.docx.pdf


In [9]:
# Create a directory for the GBIF data
gbif_dir = os.path.join(data_dir, 'gbif_redwood_dir')

In [10]:
# Permanently log into GBIF
# Do not reset credentials to avoid repeated login requests
reset_credentials = False

# Request and store username
if (not ('GBIF_USER'  in os.environ)) or reset_credentials:
    os.environ['GBIF_USER'] = input('GBIF username:')

# Request and store password
if (not ('GBIF_PWD'  in os.environ)) or reset_credentials:
    os.environ['GBIF_PWD'] = getpass('GBIF password:')
    
# Request and store email address
if (not ('GBIF_EMAIL'  in os.environ)) or reset_credentials:
    os.environ['GBIF_EMAIL'] = input('GBIF email:')

In [11]:
# Check that the login attempt was successful
'GBIF_PWD' in os.environ

True

In [12]:
# Set the species name
species_name = "Sequoia sempervirens"

# Pull the species info from GBIF
species_info = species.name_lookup(species_name, rank = 'Species')

# Grab the first result and print it
first_result = species_info['results'][0]
first_result

{'key': 127697664,
 'datasetKey': '91fecd78-0986-4713-9c36-77532846ee25',
 'nubKey': 2683909,
 'parentKey': 127697689,
 'parent': 'Sequoia',
 'kingdom': 'Plantae',
 'phylum': 'Pinophyta',
 'order': 'Pinales',
 'family': 'Taxodiaceae',
 'genus': 'Sequoia',
 'species': 'Sequoia sempervirens',
 'kingdomKey': 305476039,
 'phylumKey': 305476125,
 'classKey': 305476126,
 'orderKey': 305476127,
 'familyKey': 305476136,
 'genusKey': 127697689,
 'speciesKey': 127697664,
 'scientificName': 'Sequoia sempervirens (Lamb.) Endl.',
 'canonicalName': 'Sequoia sempervirens',
 'authorship': '(Lamb.) Endl.',
 'publishedIn': 'Syn. Conif. 198. 1847',
 'accordingTo': 'VV.AA. Flora of North America. Oxford University Press. New York and Oxford',
 'nameType': 'SCIENTIFIC',
 'taxonomicStatus': 'ACCEPTED',
 'rank': 'SPECIES',
 'origin': 'SOURCE',
 'numDescendants': 0,
 'numOccurrences': 0,
 'taxonID': '0ECE391C-C401-4F9E-B859-07B71272E97F',
 'habitats': [],
 'nomenclaturalStatus': [],
 'threatStatuses': [],
 'd

In [13]:
# Get the species key
species_key = first_result['nubKey']

# Check what the species key is
first_result['species'], species_key

('Sequoia sempervirens', 2683909)

In [14]:
# Assign the species key found in the previous inquiry
species_key = 2683909

In [18]:
# Create the file path
gbif_pattern = os.path.join(gbif_dir, '*.csv')

# Create a function to download the redwood occurrence data
if not glob(gbif_pattern):
    # Submit the query to GBIF
    gbif_query = occ.download([
        f"speciesKey = {species_key}",
        "hasCoordinate = True",
    ])
    # Only download the data once
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]
        download_key = os.environ['GBIF_DOWNLOAD_KEY']
        time.sleep(5)
    # Download the data
    download_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'],
        path = data_dir
    )
    # Unzip the file
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path = gbif_dir)

# Locate the CSV file path
gbif_path = glob(gbif_pattern)[0]

INFO:Your download key is 0043121-260226173443078
INFO:Download file size: 1453237 bytes
INFO:On disk at C:\Users\livth\earth-analytics\data\spring-03-habitat-suitability-climate-change-Livian-Von-Dran\redwood_habitat_suitability/0043121-260226173443078.zip


In [19]:
# Look at the download information
occ.download_meta("0043121-260226173443078") # Input the download key to view the information

{'key': '0043121-260226173443078',
 'doi': '10.15468/dl.5bexbs',
 'license': 'http://creativecommons.org/licenses/by-nc/4.0/legalcode',
 'request': {'predicate': {'type': 'and',
   'predicates': [{'type': 'equals',
     'key': 'SPECIES_KEY',
     'value': '2683909',
     'matchCase': False},
    {'type': 'equals',
     'key': 'HAS_COORDINATE',
     'value': 'True',
     'matchCase': False}]},
  'sendNotification': True,
  'format': 'SIMPLE_CSV',
  'type': 'OCCURRENCE',
  'verbatimExtensions': [],
  'interpretedExtensions': []},
 'created': '2026-03-17T03:24:42.070+00:00',
 'modified': '2026-03-17T03:26:35.455+00:00',
 'eraseAfter': '2026-09-17T03:24:41.922+00:00',
 'status': 'SUCCEEDED',
 'downloadLink': 'https://api.gbif.org/v1/occurrence/download/request/0043121-260226173443078.zip',
 'size': 1453237,
 'totalRecords': 21578,
 'numberDatasets': 288}

In [20]:
# Read the CSV
gbif_df = pd.read_csv(
    gbif_path,
    delimiter = '\t'
)

# Check the dataframe
gbif_df

C:\Users\livth\AppData\Local\Temp\ipykernel_7456\257810896.py:2: DtypeWarning: Columns (0: infraspecificEpithet) have mixed types. Specify dtype option on import or set low_memory=False.
  gbif_df = pd.read_csv(


,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
0,986394934,2429287b-ef65-4cfd-afcc-11cc3ba95cca,urn:catalog:ALTA-VP:132704,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,"Deyholos, M. K.",NaN,CC0_1_0,University of Alberta Museums,"Deyholos, M. K.",NaN,NaN,2026-02-27T16:46:54.642Z,NaN,GEODETIC_DATUM_ASSUMED_WGS84;OCCURRENCE_STATUS...
1,930742189,0096dfc0-9925-47ef-9700-9b77814295f1,http://bioimages.vanderbilt.edu/ind-kaufmannm/...,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,Maurice J. Kaufmann,1977-01-01T00:00:00,CC0_1_0,NaN,Maurice J. Kaufmann,NaN,native,2026-01-30T23:26:10.617Z,StillImage,NaN
2,930742172,0096dfc0-9925-47ef-9700-9b77814295f1,http://bioimages.vanderbilt.edu/ind-kaufmannm/...,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,Maurice J. Kaufmann,1977-01-01T00:00:00,CC0_1_0,NaN,Maurice J. Kaufmann,NaN,native,2026-01-30T23:26:12.158Z,StillImage,NaN
3,930742166,0096dfc0-9925-47ef-9700-9b77814295f1,http://bioimages.vanderbilt.edu/ind-kaufmannm/...,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,Maurice J. Kaufmann,1977-01-01T00:00:00,CC0_1_0,NaN,Maurice J. Kaufmann,NaN,native,2026-01-30T23:26:11.647Z,StillImage,NaN
4,930741653,0096dfc0-9925-47ef-9700-9b77814295f1,http://bioimages.vanderbilt.edu/ind-baskauf/41...,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,Steven J. Baskauf,2005-07-28T00:00:00,CC0_1_0,NaN,Steven J. Baskauf,NaN,native,2026-01-30T23:26:10.994Z,StillImage,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21573,1024187447,50c9509d-22c7-4a22-a47d-8c48425ef4a7,http://www.inaturalist.org/observations/777058,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,Ken-ichi Ueda,2014-07-10T07:56:43,CC0_1_0,Ken-ichi Ueda,Ken-ichi Ueda,NaN,NaN,2026-03-12T02:58:59Z,StillImage,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NO...
21574,1024186865,50c9509d-22c7-4a22-a47d-8c48425ef4a7,http://www.inaturalist.org/observations/775641,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,Chris Cook,2014-07-09T04:59:15,CC_BY_NC_4_0,Chris Cook,Chris Cook,NaN,NaN,2026-03-12T05:31:53.095Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NO...
21575,1024184572,50c9509d-22c7-4a22-a47d-8c48425ef4a7,http://www.inaturalist.org/observations/769678,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,JJ Johnson,2014-07-05T03:18:09,CC_BY_NC_4_0,JJ Johnson,JJ Johnson,NaN,NaN,2026-03-12T07:26:02.519Z,StillImage,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NO...
21576,1024184101,50c9509d-22c7-4a22-a47d-8c48425ef4a7,http://www.inaturalist.org/observations/767949,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,James Maughn,2014-07-04T06:20:45,CC_BY_NC_4_0,James Maughn,James Maughn,NaN,NaN,2026-03-12T05:32:12.555Z,StillImage,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NO...


In [21]:
# Look at the columns
gbif_df.columns

Index(['gbifID', 'datasetKey', 'occurrenceID', 'kingdom', 'phylum', 'class',
       'order', 'family', 'genus', 'species', 'infraspecificEpithet',
       'taxonRank', 'scientificName', 'verbatimScientificName',
       'verbatimScientificNameAuthorship', 'countryCode', 'locality',
       'stateProvince', 'occurrenceStatus', 'individualCount',
       'publishingOrgKey', 'decimalLatitude', 'decimalLongitude',
       'coordinateUncertaintyInMeters', 'coordinatePrecision', 'elevation',
       'elevationAccuracy', 'depth', 'depthAccuracy', 'eventDate', 'day',
       'month', 'year', 'taxonKey', 'speciesKey', 'basisOfRecord',
       'institutionCode', 'collectionCode', 'catalogNumber', 'recordNumber',
       'identifiedBy', 'dateIdentified', 'license', 'rightsHolder',
       'recordedBy', 'typeStatus', 'establishmentMeans', 'lastInterpreted',
       'mediaType', 'issue'],
      dtype='str')

In [22]:
# Convert the dataframe into a geodataframe (GDF)
gbif_gdf = (
    gpd.GeoDataFrame(
        gbif_df,
        # Add geometry columns to convert to a GDF
        geometry = gpd.points_from_xy(
            gbif_df.decimalLongitude,
            gbif_df.decimalLatitude
        ),
        crs = 'EPSG:4326'
    )
)

# Display the GDF data
gbif_gdf

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue,geometry
0,986394934,2429287b-ef65-4cfd-afcc-11cc3ba95cca,urn:catalog:ALTA-VP:132704,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,NaN,CC0_1_0,University of Alberta Museums,"Deyholos, M. K.",NaN,NaN,2026-02-27T16:46:54.642Z,NaN,GEODETIC_DATUM_ASSUMED_WGS84;OCCURRENCE_STATUS...,POINT (-113.51667 53.51667)
1,930742189,0096dfc0-9925-47ef-9700-9b77814295f1,http://bioimages.vanderbilt.edu/ind-kaufmannm/...,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,1977-01-01T00:00:00,CC0_1_0,NaN,Maurice J. Kaufmann,NaN,native,2026-01-30T23:26:10.617Z,StillImage,NaN,POINT (-124.0411 41.3232)
2,930742172,0096dfc0-9925-47ef-9700-9b77814295f1,http://bioimages.vanderbilt.edu/ind-kaufmannm/...,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,1977-01-01T00:00:00,CC0_1_0,NaN,Maurice J. Kaufmann,NaN,native,2026-01-30T23:26:12.158Z,StillImage,NaN,POINT (-124.0411 41.3232)
3,930742166,0096dfc0-9925-47ef-9700-9b77814295f1,http://bioimages.vanderbilt.edu/ind-kaufmannm/...,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,1977-01-01T00:00:00,CC0_1_0,NaN,Maurice J. Kaufmann,NaN,native,2026-01-30T23:26:11.647Z,StillImage,NaN,POINT (-124.0411 41.3232)
4,930741653,0096dfc0-9925-47ef-9700-9b77814295f1,http://bioimages.vanderbilt.edu/ind-baskauf/41...,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,2005-07-28T00:00:00,CC0_1_0,NaN,Steven J. Baskauf,NaN,native,2026-01-30T23:26:10.994Z,StillImage,NaN,POINT (-124.087 41.79557)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21573,1024187447,50c9509d-22c7-4a22-a47d-8c48425ef4a7,http://www.inaturalist.org/observations/777058,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,2014-07-10T07:56:43,CC0_1_0,Ken-ichi Ueda,Ken-ichi Ueda,NaN,NaN,2026-03-12T02:58:59Z,StillImage,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NO...,POINT (-123.8809 39.84946)
21574,1024186865,50c9509d-22c7-4a22-a47d-8c48425ef4a7,http://www.inaturalist.org/observations/775641,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,2014-07-09T04:59:15,CC_BY_NC_4_0,Chris Cook,Chris Cook,NaN,NaN,2026-03-12T05:31:53.095Z,NaN,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NO...,POINT (-123.71906 39.85849)
21575,1024184572,50c9509d-22c7-4a22-a47d-8c48425ef4a7,http://www.inaturalist.org/observations/769678,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,2014-07-05T03:18:09,CC_BY_NC_4_0,JJ Johnson,JJ Johnson,NaN,NaN,2026-03-12T07:26:02.519Z,StillImage,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NO...,POINT (-122.29738 37.27652)
21576,1024184101,50c9509d-22c7-4a22-a47d-8c48425ef4a7,http://www.inaturalist.org/observations/767949,Plantae,Tracheophyta,Pinopsida,Pinales,Cupressaceae,Sequoia,Sequoia sempervirens,...,2014-07-04T06:20:45,CC_BY_NC_4_0,James Maughn,James Maughn,NaN,NaN,2026-03-12T05:32:12.555Z,StillImage,CONTINENT_DERIVED_FROM_COORDINATES;TAXON_ID_NO...,POINT (-121.92512 36.99144)


In [23]:
# Create an interactive plot of the GDIF redwood observation data
gbif_gdf.hvplot(
    geo = True,
    tiles = 'EsriImagery',
    title = 'Redwood Observations on GDIF',
    # Avoid using a fill color, but select a line color of your choice
    fill_color = None,
    line_color = 'purple',
)

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Longitude,Latitude]

### Step 1b: Select study sites
Based on your research and/or range maps you find online, select at least 2 sites where your species occurs. These could be national parks, national forests, national grasslands or other protected areas, or some other area you're interested in. You can access protected area polygons from the [US Geological Survey's Protected Area Database](https://www.usgs.gov/programs/gap-analysis-project/science/pad-us-data-overview), [national grassland units](https://data.fs.usda.gov/geodata/edw/edw_resources/shp/S_USA.NationalGrassland.zip), etc.

When selecting your sites, you might want to look for places that are marginally habitable for this species, since those locations will be most likely to show changes due to climate.

Generate a site map for each location.

In [ ]:
# Create a directory for the forest sites
site_dir = Path(data_dir) / "redwood_sites"
site_dir.mkdir(parents=True, exist_ok=True)

# Create a path to access the zip file
"""Most California redwoods are found in protected state and National Parks in California, so the database of choice is  
the California Protected Areas Database (CPAD). As of 2025, the name of the zip file is "cpad_release_2025b.zip,"
downloadable at 
https://data.cnra.ca.gov/dataset/0ae3cd9f-0612-4572-8862-9e9a1c41e659/resource/27323846-4000-42a2-85b3-93ae40edeff9/download/cpad_release_2025b.zip."""
zip_path = site_dir / "cpad_release_2025b.zip"

# Print the zip file path
print(zip_path)

C:\Users\livth\earth-analytics\data\spring-03-habitat-suitability-climate-change-Livian-Von-Dran\redwood_habitat_suitability\redwood_sites\cpad_release_2025b.zip


In [38]:
# Establish the redwood site URL for the download
redwood_url = "https://data.cnra.ca.gov/dataset/0ae3cd9f-0612-4572-8862-9e9a1c41e659/resource/27323846-4000-42a2-85b3-93ae40edeff9/download/cpad_release_2025b.zip"

# Download the data only once
if not zip_path.exists():
     with requests.get(redwood_url, stream=True) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

# Extract the files
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(site_dir)

In [40]:
# Define a path to the desired shapefile
"""Individual state and National Parks will be distinct layers in the units shapefile."""
redwood_shp = site_dir / "CPAD_2025b_Units" / "CPAD_2025b_Units.shp"

# List the layers present in the shapefile
layers = fiona.listlayers(redwood_shp)
layers

INFO:GDAL signalled an error: err_no=4, msg='C:\\Users\\livth\\earth-analytics\\data\\spring-03-habitat-suitability-climate-change-Livian-Von-Dran\\redwood_habitat_suitability\\redwood_sites\\CPAD_2025b_Units\\CPAD_2025b_Units.shp: No such file or directory'


DriverError: Failed to open dataset (flags=68): C:\Users\livth\earth-analytics\data\spring-03-habitat-suitability-climate-change-Livian-Von-Dran\redwood_habitat_suitability\redwood_sites\CPAD_2025b_Units\CPAD_2025b_Units.shp

In [ ]:
### Insert your site maps here:

**Reflect and Respond**: 
Write a site description for each of your sites, or for all of your sites as a group if you have chosen a large number of linked sites. What
differences or trends in habitat suitability over time do you expect to see among your sites?

Your response here:

### Step 1c: Select time periods

In general when studying climate, we are interested in **climate
normals**, which are typically calculated from 30 years of data so that
they reflect the climate as a whole and not a single year which may be
anomalous. So if you are interested in the climate around 2050, you will need to access climate data from 2035-2065.

**Reflect and Respond**: Select at least two 30-year time periods to compare, such as historical and 30 years into the future. These time periods should help you to answer your scientific question.

Your response here:

### Step 1d: Select climate models

There is a great deal of uncertainty among the many global climate
models available. One way to work with the variety is by using an
**ensemble** of models to try to capture that uncertainty. This also
gives you an idea of the range of possible values you might expect! To
be most efficient with your time and computing resources, you can use a
subset of all the climate models available to you. However, for each
scenario, you should attempt to include models that are:

-   Warm and wet
-   Warm and dry
-   Cold and wet
-   Cold and dry

for each of your sites.

To figure out which climate models to use, you will need to access
summary data near your sites for each of the climate models. You can do
this using the [Climate Futures Toolbox Future Climate Scatter
tool](https://climatetoolbox.org/tool/Future-Climate-Scatter). There is
no need to write code to select your climate models, since this choice
is something that requires your judgement and only needs to be done
once.

If your question requires it, you can also choose to include multiple
climate variables, such as temperature and precipitation, and/or
multiple emissions scenarios, such as RCP4.5 and RCP8.5.

**Reflect and respond**: Choose at least 4 climate models that cover the range of possible future climate variability at your sites. Which models did you choose, and how did you make that decision?

Your response here (don't forget to cite the Climate Toolbox): 

## STEP 2: Data access

### Step 2a: Soil data

The [POLARIS dataset](http://hydrology.cee.duke.edu/POLARIS/) is a
convenient way to uniformly access a variety of soil parameters such as
pH and percent clay in the US. It is available for a range of depths (in
cm) and split into 1x1 degree tiles.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><p>Write a <strong>function with a numpy-style docstring</strong> that
will download POLARIS data for a particular location, soil parameter,
and soil depth. Your function should account for the situation where
your site boundary crosses over multiple tiles, and merge the necessary
data together.</p>
<p>Then, use loops to download and organize the rasters you will need to
complete this section. Include soil parameters that will help you to
answer your scientific question. We recommend using a soil depth that
best corresponds with the rooting depth of your species.</p></div></div>

In [ ]:
### Download and process soil data

In [ ]:
### Visualize the soil data

### Step 2b: Topographic data

Depending on your species habitat needs/environmental parameters, you might be interested in elevation, slope, and/or aspect. You can access reliable elevation data from the [SRTM
dataset](https://www.earthdata.nasa.gov/data/instruments/srtm),
available through the [earthaccess
API](https://earthaccess.readthedocs.io/en/latest/quick-start/). Once you have elevation data, you can calculate slope and aspect.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><p>Write a <strong>function with a numpy-style docstring</strong> that
will download SRTM elevation data for a particular location and
calculate any additional topographic variables you need such as slope or
aspect.</p>
<p>Then, use loops to download and organize the rasters you will need to
complete this section. Include topographic parameters that will help you
to answer your scientific question.</p></div></div>

> **Warning**
>
> Be careful when computing the slope from elevation that the units of
> elevation match the projection units (e.g. meters and meters, not
> meters and degrees). You will need to project the SRTM data to
> complete this calculation correctly.

In [ ]:
### Download and process topographic data

In [ ]:
### Visualize the topographic data

### Step 2c: Climate model data

You can use MACAv2 data for historical and future climate data. Be sure
to compare at least two 30-year time periods (e.g. historical vs. 10
years in the future) for at least four of the CMIP models. Overall, you
should be downloading at least 8 climate rasters for each of your sites,
for a total of 16. **You will *need* to use loops and/or functions to do
this cleanly!**.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><p>Write a <strong>function with a numpy-style docstring</strong> that
will download MACAv2 data for a particular climate model, emissions
scenario, spatial domain, and time frame. Then, use loops to download
and organize the 16+ rasters you will need to complete this section. The
<a
href="http://thredds.northwestknowledge.net:8080/thredds/reacch_climate_CMIP5_macav2_catalog2.html">MACAv2
dataset is accessible from their Thredds server</a>. Include an
arrangement of sites, models, emissions scenarios, and time periods that
will help you to answer your scientific question.</p></div></div>

In [ ]:
### Download climate data

**Reflect and respond**: Make sure to include a description of the climate data and how you selected your models. Include a citation of the MACAv2 data.

Your response here:

## STEP 3: Harmonize data
To use all your environmental and climate data layers together, you need to harmonize the different rasters you've downloaded and processed. 

As a first step, make sure that the grids for all the rasters match each other. Check out the <a href="https://corteva.github.io/rioxarray/stable/examples/reproject_match.html#Reproject-Match"><code>ds.rio.reproject_match()</code> method</a> from <code>rioxarray</code>. Make sure to use the data source that has the highest resolution as a template!</p></div></div>

> **Warning**
>
> If you are reprojecting data (as you need to here), the order of
> operations is important! Recall that reprojecting will typically tilt
> your data, leaving narrow sections of the data at the edge blank.
> However, to reproject efficiently it is best for the raster to be as
> small as possible before performing the operation. We recommend the
> following process:
>
>     1. Crop the data, leaving a buffer around the final boundary
>     2. Reproject to match the template grid (this will also crop any leftovers off the image)

In [ ]:
### Align the grids of the different data layers

## STEP 4: Develop a fuzzy logic model

A fuzzy logic model is one that is built on expert knowledge rather than
training data. You may wish to use the
[`scikit-fuzzy`](https://pythonhosted.org/scikit-fuzzy/) library, which
includes many utilities for building this sort of model. In particular,
it contains a number of **membership functions** which can convert your
data into values from 0 to 1 using information such as, for example, the
maximum, minimum, and optimal values for soil pH.

To train a fuzzy logic habitat suitability model:</p>
<pre><code>1. Find the optimal values for your species for each variable you are using (e.g. soil pH, slope, and current annual precipitation). 
2. For each **digital number** in each raster, assign a **continuous** value from 0 to 1 for how close that grid square/pixel is to the optimum range (1 = optimal, 0 = incompatible). 
3. Combine your layers by multiplying them together. This will give you a single suitability number for each grid square.
4. Optionally, you may apply a suitability threshold to make the most suitable areas pop on your map.</code></pre></div></div>

> **Tip**
>
> If you use mathematical operators on a raster in Python, it will
> automatically perform the operation for every number in the raster.
> This type of operation is known as a **vectorized** function. **DO NOT
> DO THIS WITH A LOOP!**. A vectorized function that operates on the
> whole array at once will be much easier and faster.

In [ ]:
### Create fuzzy logic model for habitat suitability

## STEP 5: Present your results
Generate some plots that show your key findings of habitat suitability in your study sites across the different time periods and climate models. Don’t forget to interpret your plots!

In [ ]:
### Create plots

Interpret your plots here: